# Experimentos completos — CPCV vs WF vs KFold

**24 reales** (4 activos × 2 timelines × 3 métodos → aquí 1 experimento/row por activo × timeline)
**60 sintéticos** (20 escenarios × 3 métodos → 1 experimento/row por escenario)

Métricas de evaluación por experimento:
- **delta** = μ(val_SRs) − SR_holdout
- **bias** = delta / |SR_holdout|
- **z_score** = (SR_holdout − μ(val_SRs)) / σ(val_SRs)  ← solo meaningful para CPCV (n=15)

Plots por experimento:
1. Violin OOS por split/fold (CPCV 15 pts, WF 3 pts, KFold 3 pts)
2. Distribución paths CPCV + WF/KFold punto único + línea hold-out

Plot global (una sola vez): split-matrix CPCV 6,2

Salidas en `resultados/real/<timeline>/<ticker>/` y `resultados/synthetic/<scenario>/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from xgboost import XGBClassifier

from cpcv_analysis.config import (
    XGB_PARAMS, ASSETS, TIMELINES, SYNTHETIC_SCENARIOS, TIMELINE_B, N_GROUPS, K_TEST
)
from cpcv_analysis.experiment import run_experiment_full, run_experiment_full_from_arrays
from cpcv_analysis.synthetic import generate_synthetic_prices
from cpcv_analysis.data import build_features
from cpcv_analysis.plots import (
    plot_split_matrix,
    plot_fold_oos_violins,
    plot_paths_vs_holdout,
)
from cpcv_analysis.splitters import CombinatorialPurgedKFold
from cpcv_analysis.data import load_asset
from cpcv_analysis.backtest_engine import slice_by_dates

clf_template = XGBClassifier(**XGB_PARAMS)
ASSET_TICKERS = list(ASSETS.keys())

# ── Directorios de salida ─────────────────────────────────────────────────────
os.makedirs('resultados', exist_ok=True)
for tl in TIMELINES:
    for ticker in ASSET_TICKERS:
        os.makedirs(f"resultados/real/{tl['name']}/{ticker}", exist_ok=True)
for sc in SYNTHETIC_SCENARIOS:
    os.makedirs(f"resultados/synthetic/{sc['id']}_{sc['name']}", exist_ok=True)

print(f'Activos: {ASSET_TICKERS}')
print(f'Timelines: {[t["name"] for t in TIMELINES]}')
print('Directorios listos.')

## Split matrix CPCV 6,2 (una sola vez)
Muestra qué grupos actúan como test en cada uno de los 15 splits.

In [ ]:
# Construir tabla de splits para cualquier dataset de tamaño similar (usamos SPY timeline A)
from cpcv_analysis.data import load_asset
from cpcv_analysis.config import TIMELINE_A

_X, _y, _t1, _, _fr = load_asset('SPY', TIMELINE_A['download_start'], TIMELINE_A['download_end'])
_X_dev, _y_dev, _t1_dev, _fr_dev = slice_by_dates(
    _X, _y, _t1, _fr, TIMELINE_A['dev_start'], TIMELINE_A['dev_end'])

_cpcv = CombinatorialPurgedKFold(N_GROUPS, K_TEST, _t1_dev, 0.01)
_split_table = []
for sid, (_, _, _, test_groups, _, _) in enumerate(_cpcv.split(_X_dev)):
    _split_table.append({'split_id': sid, 'test_groups': test_groups})
_split_df = pd.DataFrame(_split_table)

plot_split_matrix(_split_df, N=N_GROUPS, out_dir='resultados/')
from IPython.display import Image
Image('resultados/02_split_matrix.png')

## Experimentos reales (4 activos × 2 timelines)

In [ ]:
real_rows = []

for timeline_cfg in TIMELINES:
    for ticker in ASSET_TICKERS:
        out_dir = f"resultados/real/{timeline_cfg['name']}/{ticker}/"
        print(f"\n{'='*60}")
        print(f"Corriendo: {timeline_cfg['name']} | {ticker}")
        try:
            res = run_experiment_full(ticker, timeline_cfg, clf_template)

            # ── Violin OOS por split/fold ──────────────────────────────────
            label = f"{ticker} {timeline_cfg['name']}"
            plot_fold_oos_violins(
                cpcv_fold_srs=res['cpcv_fold_srs'],
                wf_fold_srs=res['wf_fold_srs'],
                kfold_fold_srs=res['kfold_fold_srs'],
                label=label,
                out_dir=out_dir,
            )

            # ── Paths vs holdout ──────────────────────────────────────────
            plot_paths_vs_holdout(
                cpcv_path_srs=res['cpcv_path_srs'],
                wf_sr=res['wf_sr'],
                kfold_sr=res['kfold_sr'],
                holdout_sr=res['holdout_sr'],
                label=label,
                out_dir=out_dir,
            )

            # ── Recolectar métricas ───────────────────────────────────────
            for method in ['cpcv', 'wf', 'kfold']:
                m = res[f'metrics_{method}']
                row = dict(
                    timeline=timeline_cfg['name'],
                    asset=ticker,
                    method=method,
                    holdout_sr=res['holdout_sr'],
                    **m,
                )
                # z_score solo es meaningful para cpcv (n=15 paths)
                if method != 'cpcv':
                    row['z_score'] = float('nan')
                real_rows.append(row)
                print(f"  {method}: delta={m['delta']:.3f}  bias={m['bias']:.3f}  "
                      f"z={m.get('z_score', float('nan')):.2f}  holdout_sr={res['holdout_sr']:.3f}")

        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  ERROR: {e}')

real_df = pd.DataFrame(real_rows)
real_df.to_csv('resultados/real_experiments.csv', index=False)
print(f'\nCompletados {len(real_rows)} filas reales (esperado 24)')

## Tabla reales — todas las filas

In [ ]:
display_cols = ['timeline', 'asset', 'method', 'holdout_sr', 'delta', 'bias', 'z_score',
                'coverage_90', 'rank_pct', 'dispersion', 'pct_positive']
display(real_df[display_cols].round(4))

## Tabla reales — agregada por método

In [ ]:
agg_cols = ['delta', 'bias', 'z_score', 'coverage_90', 'rank_pct', 'dispersion', 'pct_positive']
real_summary = real_df.groupby('method')[agg_cols].mean().round(4)
real_summary.index.name = 'method'
real_summary.to_csv('resultados/real_summary_by_method.csv')

print('=== Métricas promedio por método (24 experimentos reales) ===')
display(real_summary)

## Experimentos sintéticos (20 escenarios × 3 métodos)

In [ ]:
synth_rows = []

for scenario in SYNTHETIC_SCENARIOS:
    seed = int(scenario['id'])
    prices_synth = generate_synthetic_prices(scenario, seed=seed)
    X_s, y_s, t1_s, _, fwd_ret_s = build_features(prices_synth)

    timeline_synth = dict(
        name=f"synth_{scenario['id']}",
        wf_start=TIMELINE_B['wf_start'],
        dev_start=TIMELINE_B['dev_start'],
        dev_end=TIMELINE_B['dev_end'],
        retrain_start=TIMELINE_B['retrain_start'],
        retrain_end=TIMELINE_B['retrain_end'],
        holdout_start=TIMELINE_B['holdout_start'],
        holdout_end=TIMELINE_B['holdout_end'],
        download_start=TIMELINE_B['download_start'],
        download_end=TIMELINE_B['download_end'],
    )

    out_dir = f"resultados/synthetic/{scenario['id']}_{scenario['name']}/"
    print(f"\nSintético {scenario['id']} {scenario['name']}")
    try:
        res = run_experiment_full_from_arrays(
            X_s, y_s, t1_s, prices_synth, fwd_ret_s, timeline_synth, clf_template)

        label = f"{scenario['id']}_{scenario['name']}"
        plot_fold_oos_violins(
            cpcv_fold_srs=res['cpcv_fold_srs'],
            wf_fold_srs=res['wf_fold_srs'],
            kfold_fold_srs=res['kfold_fold_srs'],
            label=label,
            out_dir=out_dir,
        )
        plot_paths_vs_holdout(
            cpcv_path_srs=res['cpcv_path_srs'],
            wf_sr=res['wf_sr'],
            kfold_sr=res['kfold_sr'],
            holdout_sr=res['holdout_sr'],
            label=label,
            out_dir=out_dir,
        )

        for method in ['cpcv', 'wf', 'kfold']:
            m = res[f'metrics_{method}']
            row = dict(
                scenario_id=scenario['id'],
                scenario_name=scenario['name'],
                method=method,
                holdout_sr=res['holdout_sr'],
                **m,
            )
            if method != 'cpcv':
                row['z_score'] = float('nan')
            synth_rows.append(row)
            print(f"  {method}: delta={m['delta']:.3f}  bias={m['bias']:.3f}  "
                  f"z={m.get('z_score', float('nan')):.2f}  holdout_sr={res['holdout_sr']:.3f}")

    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  ERROR: {e}')

synth_df = pd.DataFrame(synth_rows)
synth_df.to_csv('resultados/synthetic_experiments.csv', index=False)
print(f'\nCompletados {len(synth_rows)} filas sintéticas (esperado 60)')

## Tabla sintéticas — todas las filas

In [ ]:
display_cols2 = ['scenario_id', 'scenario_name', 'method', 'holdout_sr',
                 'delta', 'bias', 'z_score', 'coverage_90', 'rank_pct', 'dispersion', 'pct_positive']
display(synth_df[display_cols2].round(4))

## Tabla sintéticas — agregada por método

In [ ]:
agg_cols = ['delta', 'bias', 'z_score', 'coverage_90', 'rank_pct', 'dispersion', 'pct_positive']
synth_summary = synth_df.groupby('method')[agg_cols].mean().round(4)
synth_summary.index.name = 'method'
synth_summary.to_csv('resultados/synthetic_summary_by_method.csv')

print('=== Métricas promedio por método (60 experimentos sintéticos) ===')
display(synth_summary)

## Verificación final

In [ ]:
import glob
real_plots  = glob.glob('resultados/real/**/*.png', recursive=True)
synth_plots = glob.glob('resultados/synthetic/**/*.png', recursive=True)

print(f'Filas reales:      {len(real_df)} (esperado 24)')
print(f'Filas sintéticas:  {len(synth_df)} (esperado 60)')
print(f'Plots reales:      {len(real_plots)}')
print(f'Plots sintéticos:  {len(synth_plots)}')
print(f'CSVs en resultados/: {sorted(os.listdir("resultados/"))}' )

assert len(real_df) == 24, f'Esperado 24, got {len(real_df)}'
assert len(synth_df) == 60, f'Esperado 60, got {len(synth_df)}'
print('Todas las aserciones pasaron.')